# BTXRD Bone-Tumor Segmentation - Colab Experiment
### DenseNet121 -> LayerCAM -> Tumor Morphology -> SAM ViT-B -> Evaluation / U-Net

This notebook is an executable experiment report for the **BTXRD** dataset, designed to
run on **Google Colab** with the project and dataset stored on Google Drive. It mirrors
`btxrd_kaggle.ipynb` (Kaggle) and `thesis_experiment_debug.ipynb` (Colab, RAM-H1200)
but targets bone-**tumor** segmentation.

Every major stage includes:

- the purpose of the stage;
- its inputs and outputs;
- the exact command being executed;
- visual checks for image-processing outputs.

Dataset: **BTXRD only** (Bone Tumor X-Ray Dataset, 3,746 hand/limb/pelvis radiographs,
1,867 with a tumor). The segmentation target is the **tumor lesion region**, not the
whole bone or the hand/limb silhouette. The image-level label used to train the
classifier and drive LayerCAM is the `tumor` column (tumor vs normal) — this keeps the
pipeline a true weakly-supervised segmentation setup: SAM and the morphology module
never see the ground-truth polygon/bbox annotations, which are reserved for evaluation
only.

BTXRD ships with **no predefined train/val/test split**. This project derives one
locally with an 80/10/10 stratified split (by normal/benign/malignant) using a fixed
seed, so results are reproducible across runs (see `datasets/btxrd.py`).

**Before running:** download BTXRD from
[kaggle.com/datasets/thanhngan123/btxrd-data](https://www.kaggle.com/datasets/thanhngan123/btxrd-data)
and upload it to your Google Drive at `Thesis Experiment/BTXRD/` (containing
`images/`, `Annotations/`, and `dataset.csv` or `dataset.xlsx`), matching the layout
`thesis_experiment_debug.ipynb` already uses for the repo checkout.

## 0. Pipeline Overview

```text
BTXRD hand/limb/pelvis X-ray
  -> Stage 1: DenseNet121 tumor-vs-normal checkpoint
  -> Stage 2: LayerCAM localization
  -> Stage 3: tumor-specific morphology (CAM-dominant, local-anomaly aware)
  -> Stage 4: SAM box/point prompting
  -> Stage 5: bone-aware mask selection and conservative refinement
  -> pseudo tumor mask
  -> Dice/IoU against BTXRD ground truth (LabelMe polygon masks)

Optional baseline:
BTXRD image + GT tumor mask -> U-Net -> supervised segmentation result
```

BTXRD provides ground-truth tumor polygons (converted to binary masks by
`datasets/btxrd.py`), so the pseudo-mask branch can be evaluated quantitatively.

**Why a separate tumor morphology module?** `pseudo/bone_morphology.py` was tuned for
RAM-H1200, where the classifier only has a whole-hand label, so CAM is a weak anchor
and the priority is finding *radiopaque bone* vs soft tissue. BTXRD's classifier is
trained directly on tumor presence, so its CAM is a much stronger localization signal.
`pseudo/tumor_morphology.py` weights CAM more heavily and looks for local intensity
*anomalies* (both lytic/dark and sclerotic/bright lesions) instead of assuming the
target is always the brightest tissue. `pseudo/morphology_factory.py` selects between
the two modules via `--dataset`.

## 1. Google Drive Paths and Run Switches

This cell mounts Google Drive, then defines repository paths, output folders, image
size, and which expensive stages should run.

Default full run:

- keep `RUN_TRAIN_CLASSIFIER=True` if no compatible `best_classifier.pt` is saved on
  Drive yet;
- `RUN_FULL_PSEUDO_MASKS=True` generates pseudo masks for the full validation split;
- visualization/debug cells still show only a few samples so the notebook remains
  readable.

Adjust `WORKSPACE_ROOT` if your Drive layout differs from
`Thesis Experiment/{repo/project, BTXRD}`.

**Tuning workflow:** Stage 2 (pseudo-mask generation) is the slowest stage — SAM alone
adds ~0.1-0.5s per prediction, on top of classifier + LayerCAM + morphology per image.
When iterating on parameters (percentile thresholds, `--sam-prompt-mode`,
`--selection-method`, etc.), set `RUN_FULL_PSEUDO_MASKS = False` first: this runs only
15 images (`--max-images 15`), enough to sanity-check CAM overlays, pseudo masks, and
`prompt_quality.csv` (section 7.3) without waiting on the full split. Only flip back to
`True` once parameters are settled and you need the official Dice/IoU numbers over the
full validation split for the report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys
import subprocess

# WORKSPACE_ROOT is the project root on Google Drive, matching the layout already
# used by thesis_experiment_debug.ipynb.
WORKSPACE_ROOT = Path('/content/drive/MyDrive/Thesis')
PROJECT_DIR = WORKSPACE_ROOT / 'repo' / 'project'
BTXRD_ROOT_DEFAULT = WORKSPACE_ROOT / 'BTXRD'

REPO_URL = 'https://github.com/itsthang333/Thesis.git'
BRANCH = 'main'

OUTPUT_ROOT = WORKSPACE_ROOT / 'BTXRDOutputs'

# SAM_VERSION: 'v1' = original SAM (ViT-B) | 'v2' = SAM2 | 'medsam2' = SAM2
# fine-tuned on medical imagery (bowang-lab/MedSAM2). All three share the same
# point/box prompt API in this project's sam_refine.py, so this is the only
# switch needed — checkpoint/package differ and are handled below and in the
# environment-setup cell. NOTE: 'v2' and 'medsam2' both import a package named
# `sam2`, but from different repos with different vendored configs — they
# cannot coexist in one environment; switching between them requires
# reinstalling the matching package (the environment-setup cell below does
# this automatically based on this variable).
SAM_VERSION = 'v1'

# SAM2_MODEL_SIZE: 'tiny' | 'small' | 'base_plus' | 'large'. Only used when
# SAM_VERSION == 'v2' (MedSAM2 only ships a tiny-based checkpoint). Larger
# models are slower per image but may produce better candidate masks — the
# oracle diagnostic in section 7.4 shows whether SAM is the bottleneck (a
# larger model helps) or mask_selection.py is (it won't). tiny is the
# fastest/smallest and, in this project's own ablation, scored *better* than
# small on BTXRD — bigger is not automatically better on this domain.
SAM2_MODEL_SIZE = 'tiny'
_SAM2_CFG_SUFFIX = {'tiny': 't', 'small': 's', 'base_plus': 'b+', 'large': 'l'}[SAM2_MODEL_SIZE]

if SAM_VERSION == 'v1':
    SAM2_MODEL_CFG = None
    SAM_CHECKPOINT = OUTPUT_ROOT / 'checkpoints' / 'sam_vit_b_01ec64.pth'
elif SAM_VERSION == 'v2':
    SAM2_MODEL_CFG = f'configs/sam2.1/sam2.1_hiera_{_SAM2_CFG_SUFFIX}.yaml'
    SAM_CHECKPOINT = OUTPUT_ROOT / 'checkpoints' / f'sam2.1_hiera_{SAM2_MODEL_SIZE}.pt'
else:  # medsam2
    SAM2_MODEL_CFG = 'configs/sam2.1_hiera_t512.yaml'
    SAM_CHECKPOINT = OUTPUT_ROOT / 'checkpoints' / 'MedSAM2_latest.pt'

# FUSION_TOPK controls select_and_fuse_masks' fusion mode in mask_selection.py,
# but ONLY takes effect when DISABLE_BEST_PER_COMPONENT=True below. With the
# default morphology-fusion-mode=components, generate_pseudo_masks.py always
# passes best_per_component=True whenever component_ids exist, and that
# branch in select_and_fuse_masks returns before fusion_topk's own branch is
# ever reached — so changing FUSION_TOPK alone is a no-op in the default
# configuration. (Confirmed by testing: FUSION_TOPK=1 produced byte-identical
# output to FUSION_TOPK=3 on the same MedSAM2 run.)
#   1     = keep only the single best-scoring SAM candidate (no union)
#   3     = union (logical-OR) of the top-3 above-threshold candidates
FUSION_TOPK = 3

# DISABLE_BEST_PER_COMPONENT: best_per_component (the default) unions the
# single best SAM candidate from *every* kept morphology component (up to
# MAX_BONE_COMPONENTS below) into the final mask. If a component_id belongs
# to bone texture/edge noise rather than the real lesion, its best candidate
# still gets unioned in -- invisible to oracle_best_single_dice (which only
# ever looks at ONE candidate at a time) and not affected by FUSION_TOPK at
# all. Set this to True to instead use fusion_topk's global top-k selection
# across all candidates (ignoring which component each came from) -- this is
# the actual A/B test for whether per-component unioning of non-lesion
# components is what drives a large oracle_gap_dice.
DISABLE_BEST_PER_COMPONENT = False

CLASSIFIER_OUTPUT = OUTPUT_ROOT / 'classifier'
PSEUDO_OUTPUT = OUTPUT_ROOT / 'pseudo_masks'
SEG_OUTPUT = OUTPUT_ROOT / 'segmentation'
EVAL_CSV = OUTPUT_ROOT / 'btxrd_eval.csv'
VIZ_OUTPUT = OUTPUT_ROOT / 'viz'
DEBUG_OUTPUT = OUTPUT_ROOT / 'debug_viz'
MORPH_DEBUG_OUTPUT = OUTPUT_ROOT / 'morphology_debug'
ABLATION_OUTPUT = OUTPUT_ROOT / 'ablation_preview'

DATASET_NAME = 'btxrd'
IMAGE_SIZE = 384
NUM_WORKERS = 2
BATCH_SIZE_CLASSIFIER = 8
BATCH_SIZE_SEGMENTATION = 4
EPOCHS_CLASSIFIER = 25
EPOCHS_SEGMENTATION = 25

RUN_TRAIN_CLASSIFIER = True
RUN_FULL_PSEUDO_MASKS = True
RUN_EVALUATE_PSEUDO = True
RUN_TRAIN_UNET = False
RUN_VISUALIZE_SAMPLE = True
RUN_MORPHOLOGY_EXPLAINER = True
RUN_DEBUG_SAM = True
RUN_ABLATION_PREVIEW = False

# Set this if BTXRD lives somewhere else on Drive; leave '' to use BTXRD_ROOT_DEFAULT.
DATASET_OVERRIDE = ''

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

print('PROJECT_DIR:', PROJECT_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('SAM_VERSION:', SAM_VERSION, '| SAM_CHECKPOINT:', SAM_CHECKPOINT)
print('FUSION_TOPK:', FUSION_TOPK, '| DISABLE_BEST_PER_COMPONENT:', DISABLE_BEST_PER_COMPONENT)

## 2. Colab Environment Setup

This stage prepares the runtime:

1. clone the repository onto Drive (skipped if already checked out);
2. install the extra packages needed by the project, including `pandas`/`openpyxl`
   for BTXRD's `dataset.xlsx`;
3. verify GPU availability (Colab's default PyTorch build works out of the box, so
   this notebook skips the Kaggle P100/CUDA-11.8 workaround);
4. verify that the expected project scripts exist.

In [ ]:
if not PROJECT_DIR.exists():
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR.parent)], check=True)
else:
    print('Project already exists:', PROJECT_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('cwd:', Path.cwd())

def run_streaming(cmd, **kwargs):
    """Run a script subprocess and stream its stdout/stderr live into the notebook.

    Without this, tqdm progress bars and print() calls from the child process can be
    fully buffered and only appear after the process exits, since its stdout is a pipe
    rather than a real terminal. Inserting `-u` right after the interpreter forces
    unbuffered mode in the child, and reading line-by-line here flushes each line to
    the notebook as soon as it is produced.
    """
    if cmd and cmd[0] == sys.executable:
        cmd = [cmd[0], '-u', *cmd[1:]]
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        **kwargs,
    )
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    return return_code


def run_pip(args, description):
    print(f'Installing {description}...')
    run_streaming([sys.executable, '-m', 'pip', 'install', *args])
    print(f'Installed: {description}')

run_pip(['pandas', 'openpyxl'], 'pandas + openpyxl (dataset.xlsx reader)')
run_pip(['opencv-python'], 'opencv-python')

# Only install the SAM package actually selected above (SAM2/MedSAM2 require
# Python >=3.10 and a separate package — Colab's default runtime is Python
# 3.12, so this works out of the box there, but would fail on an older local
# Python). 'v2' and 'medsam2' both provide a package named `sam2` from
# different repos with different vendored configs, so they cannot coexist —
# switching SAM_VERSION between them in an existing Colab session requires
# restarting the runtime (Runtime > Restart session) before re-running this
# cell, so pip does not just silently keep the previously installed one.
if SAM_VERSION == 'v1':
    run_pip(['--no-deps', 'git+https://github.com/facebookresearch/segment-anything.git'], 'segment-anything')
elif SAM_VERSION == 'v2':
    run_pip(['git+https://github.com/facebookresearch/sam2.git'], 'sam2 (SAM2ImagePredictor)')
else:  # medsam2
    # MedSAM2's setup.py ships no MANIFEST.in and doesn't declare configs/*.yaml
    # as package_data, so a plain `pip install git+...` silently drops those
    # files from the installed package — build_sam2() then fails with
    # hydra.errors.MissingConfigException even though the .yaml is real in the
    # git repo. Cloning and installing with `pip install -e .` keeps the
    # cloned checkout (including configs/) on disk and importable, which is
    # what sam_refine.py's Hydra fallback (_build_sam2_with_config_fallback)
    # searches for as a second line of defense.
    medsam2_src = OUTPUT_ROOT / 'medsam2_src'
    if not medsam2_src.exists():
        run_streaming(['git', 'clone', 'https://github.com/bowang-lab/MedSAM2.git', str(medsam2_src)])
    run_pip(['-e', str(medsam2_src)], 'MedSAM2 (editable install, vendored sam2 + SAM2ImagePredictor)')

import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))
    print('CUDA runtime:', torch.version.cuda)
else:
    print('No GPU detected. In Colab, set Runtime > Change runtime type > GPU before training.')

required_files = [
    PROJECT_DIR / 'datasets' / 'btxrd.py',
    PROJECT_DIR / 'pseudo' / 'tumor_morphology.py',
    PROJECT_DIR / 'pseudo' / 'morphology_factory.py',
    PROJECT_DIR / 'train_classifier.py',
    PROJECT_DIR / 'generate_pseudo_masks.py',
    PROJECT_DIR / 'evaluate_ramh1200_masks.py',
    PROJECT_DIR / 'train_segmentation.py',
    PROJECT_DIR / 'visualize_pipeline.py',
]
for path in required_files:
    print(('OK  ' if path.exists() else 'MISS'), path)

## 3. Resolve BTXRD Dataset

The notebook looks for BTXRD under `BTXRD_ROOT_DEFAULT` (or `DATASET_OVERRIDE` if set)
on Google Drive. Upload BTXRD there before running this cell:
[kaggle.com/datasets/thanhngan123/btxrd-data](https://www.kaggle.com/datasets/thanhngan123/btxrd-data).

Expected layout:

```text
Thesis Experiment/BTXRD/
  images/
  Annotations/
  dataset.csv or dataset.xlsx
```

In [ ]:
from datasets.btxrd import resolve_btxrd_root

def find_btxrd_root(base: Path):
    candidates = []
    if base and base.exists():
        candidates.append(base)
        candidates.extend(base.glob('*'))
        candidates.extend(base.glob('*/*'))
    for candidate in candidates:
        try:
            return resolve_btxrd_root(candidate)
        except FileNotFoundError:
            continue
    return None

search_base = Path(DATASET_OVERRIDE) if DATASET_OVERRIDE else BTXRD_ROOT_DEFAULT
BTXRD_ROOT = find_btxrd_root(search_base)

if BTXRD_ROOT is None:
    raise FileNotFoundError(
        f'BTXRD was not found under {search_base}. Upload the dataset '
        '(images/, Annotations/, dataset.csv or dataset.xlsx) to that path on '
        'Google Drive, or set DATASET_OVERRIDE to the correct folder.'
    )

print('BTXRD_ROOT:', BTXRD_ROOT)

## 4. Dataset Inspection and Ground-Truth Visualization

**Purpose:** verify that the notebook reads the correct dataset, that the derived
80/10/10 split is stratified, and that the annotation target is the tumor lesion (not
the whole bone/limb).

**Input:** BTXRD `images/`, `Annotations/` (LabelMe JSON), and `dataset.csv`/`.xlsx`.

**Visual output:** original X-ray, GT tumor mask, and GT overlay, for both a tumor case
and a normal case.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

from datasets.btxrd import BTXRDSegmentationDataset, load_btxrd_records, split_btxrd_records

records = load_btxrd_records(BTXRD_ROOT)
print('Total BTXRD images:', len(records))
for split in ['train', 'val', 'test']:
    subset = split_btxrd_records(records, split)
    n_tumor = sum(r['tumor'] for r in subset)
    n_malignant = sum(r['malignant'] for r in subset)
    print(f'{split:<6} n={len(subset):>5} tumor={n_tumor:>4} malignant={n_malignant:>4}')

preview_ds = BTXRDSegmentationDataset(root=BTXRD_ROOT, split='val', image_size=IMAGE_SIZE)

# Find one tumor sample and one normal sample in the val split for a side-by-side check.
tumor_index = next(i for i, s in enumerate(preview_ds.samples) if s['tumor'])
normal_index = next(i for i, s in enumerate(preview_ds.samples) if not s['tumor'])

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for row, index, label in [(0, tumor_index, 'Tumor'), (1, normal_index, 'Normal')]:
    image_tensor, mask_tensor, image_name = preview_ds[index]
    image_path = preview_ds.images_dir / image_name
    original = Image.open(image_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    gt_mask = mask_tensor[0].numpy() > 0.5
    original_np = np.array(original)
    overlay = original_np.copy()
    overlay[gt_mask] = (0.55 * overlay[gt_mask] + 0.45 * np.array([255, 40, 40])).astype(np.uint8)

    axes[row][0].imshow(original_np)
    axes[row][0].set_title(f'{label}: {image_name}')
    axes[row][1].imshow(gt_mask, cmap='gray')
    axes[row][1].set_title('Ground-truth tumor mask')
    axes[row][2].imshow(overlay)
    axes[row][2].set_title('GT mask overlay')
    for ax in axes[row]:
        ax.axis('off')
plt.tight_layout()
plt.show()

## 5. SAM Checkpoint

**Purpose:** SAM ViT-B is used in Stage 2 to refine component boxes and points into
candidate masks.

If the checkpoint is not attached, this cell downloads it.

In [ ]:
SAM_CHECKPOINT_URLS = {
    'v1': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
    'v2': f'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_{SAM2_MODEL_SIZE}.pt',
    'medsam2': 'https://huggingface.co/wanglab/MedSAM2/resolve/main/MedSAM2_latest.pt',
}

if not SAM_CHECKPOINT.exists():
    label = SAM_VERSION if SAM_VERSION == 'v1' else f'{SAM_VERSION} ({SAM2_MODEL_SIZE})' if SAM_VERSION == 'v2' else SAM_VERSION
    print(f'SAM checkpoint was not found. Downloading SAM ({label})...')
    import urllib.request
    SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(SAM_CHECKPOINT_URLS[SAM_VERSION], SAM_CHECKPOINT)
else:
    print('SAM checkpoint exists:', SAM_CHECKPOINT)

## 6. Stage 1: DenseNet121 Tumor-vs-Normal Checkpoint

**Pipeline role:** provide features and gradients for LayerCAM.

**Input:** BTXRD radiographs (hand, limb, and pelvis regions).

**Training target:** `tumor` (1 = tumor present, 0 = normal), matching the WSSS
convention used for RAM-H1200's `hand` label — the classifier only ever sees a
whole-image label, never the polygon/bbox annotations.

**Output:** `best_classifier.pt` and classifier training curves.

In [ ]:
# Epochs at which to snapshot LayerCAM on a fixed set of validation images, so CAM
# localization quality can be compared visually across training — see cell below.
CAM_PREVIEW_EPOCHS = '2,4,8,15,25'
CAM_PREVIEW_COUNT = 4

# Stop early if val_f1 does not improve for this many consecutive epochs (0 disables).
CLASSIFIER_EARLY_STOP_PATIENCE = 7

classifier_cmd = [
    sys.executable, 'train_classifier.py',
    '--dataset', DATASET_NAME,
    '--ram-root', str(BTXRD_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--target-columns', 'tumor',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_CLASSIFIER),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_CLASSIFIER),
    '--output-dir', str(CLASSIFIER_OUTPUT),
    '--save-cam-epochs', CAM_PREVIEW_EPOCHS,
    '--cam-preview-count', str(CAM_PREVIEW_COUNT),
    '--early-stop-patience', str(CLASSIFIER_EARLY_STOP_PATIENCE),
]
print(' '.join(classifier_cmd))
if RUN_TRAIN_CLASSIFIER:
    run_streaming(classifier_cmd)
else:
    print('RUN_TRAIN_CLASSIFIER=False; classifier training is skipped.')

In [ ]:
import pandas as pd

CLASSIFIER_CHECKPOINT = CLASSIFIER_OUTPUT / 'best_classifier.pt'
log_path = CLASSIFIER_OUTPUT / 'training_log.csv'
print('Classifier checkpoint:', CLASSIFIER_CHECKPOINT, 'exists=', CLASSIFIER_CHECKPOINT.exists())
if log_path.exists():
    df = pd.read_csv(log_path)
    display(df.tail())
    print(f"Best val_f1: {df['val_f1'].max():.4f} (epoch {int(df['val_f1'].idxmax()) + 1})")
    print(f"Best val_acc: {df['val_acc'].max():.4f} (epoch {int(df['val_acc'].idxmax()) + 1})")
    best_row = df.loc[df['val_f1'].idxmax()]
    print(
        f"Confusion matrix at best epoch: TP={int(best_row['val_tp'])} FP={int(best_row['val_fp'])} "
        f"FN={int(best_row['val_fn'])} TN={int(best_row['val_tn'])}"
    )
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    df[['train_loss', 'val_loss']].plot(ax=axes[0], title='Classifier loss')
    df[['train_acc', 'val_acc']].plot(ax=axes[1], title='Classifier accuracy')
    df[['train_f1', 'val_f1']].plot(ax=axes[2], title='Classifier F1 (tumor class)')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Classifier training_log.csv was not found yet.')

### 6.1 CAM Quality Across Training Epochs

**Purpose:** in WSSS, classifier F1 is only a proxy — what actually matters for the
downstream pseudo-mask is whether LayerCAM localizes the tumor well. This cell shows the
same fixed validation images' CAM overlay at each snapshot epoch (`CAM_PREVIEW_EPOCHS`
above), so localization quality can be judged visually rather than assumed from F1 alone.

If CAM sharpens around the tumor region as training progresses, the classifier
checkpoint is likely usable for Stage 2 even if F1 plateaus early. If CAM stays diffuse
or drifts to unrelated regions (soft tissue, image borders), a better checkpoint or
different hyperparameters may be needed regardless of what F1 says.

In [ ]:
cam_dir = CLASSIFIER_OUTPUT / 'cam_preview'
cam_files = sorted(cam_dir.glob('cam_epoch*.png')) if cam_dir.exists() else []
print('CAM preview files:', len(cam_files), cam_dir)

if cam_files:
    # Group by sample stem (the part after 'cam_epoch###_') so each row is one
    # validation image progressing left-to-right across snapshot epochs.
    from collections import defaultdict
    import re

    by_sample = defaultdict(list)
    for path in cam_files:
        match = re.match(r'cam_epoch(\d+)_(.+)\.png', path.name)
        if not match:
            continue
        epoch_num, sample_stem = int(match.group(1)), match.group(2)
        by_sample[sample_stem].append((epoch_num, path))

    n_samples = len(by_sample)
    n_epochs = max(len(v) for v in by_sample.values())
    fig, axes = plt.subplots(n_samples, n_epochs, figsize=(3.2 * n_epochs, 3.2 * n_samples))
    if n_samples == 1:
        axes = np.array([axes])
    if n_epochs == 1:
        axes = axes.reshape(n_samples, 1)

    for row, (sample_stem, entries) in enumerate(sorted(by_sample.items())):
        entries.sort(key=lambda item: item[0])
        for col, (epoch_num, path) in enumerate(entries):
            axes[row][col].imshow(Image.open(path))
            axes[row][col].set_title(f'{sample_stem}\nepoch {epoch_num}')
            axes[row][col].axis('off')
        for col in range(len(entries), n_epochs):
            axes[row][col].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No CAM preview files found. Set CAM_PREVIEW_EPOCHS above and re-run training.')

## 7. Stage 2: Pseudo-Mask Generation with LayerCAM + Tumor Morphology + SAM

**Pipeline role:** generate a tumor pseudo mask from an X-ray image.

**Internal stages:**

1. DenseNet121 produces gradients and features from the tumor-vs-normal checkpoint.
2. LayerCAM localizes the region driving the tumor prediction.
3. Tumor morphology combines CAM (dominant), local intensity anomaly, and edge
   response — see `pseudo/tumor_morphology.py`.
4. Connected components generate boxes and structured positive points.
5. SAM proposes candidate masks.
6. `bone_hybrid` scoring selects the best masks (same scorer as RAM-H1200; it only
   consumes generic bone/tumor-likelihood + CAM + SAM-quality signals, so it transfers
   without changes).
7. Conservative morphology cleans the final pseudo mask.

In [ ]:
if not CLASSIFIER_CHECKPOINT.exists():
    raise FileNotFoundError(f'Missing classifier checkpoint: {CLASSIFIER_CHECKPOINT}. Run Stage 1 first.')

process_args = ['--process-all'] if RUN_FULL_PSEUDO_MASKS else ['--max-images', '15']
pseudo_cmd = [
    sys.executable, 'generate_pseudo_masks.py',
    '--dataset', DATASET_NAME,
    '--ram-root', str(BTXRD_ROOT),
    '--split', 'val',
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--sam-version', SAM_VERSION,
    *(['--sam2-model-cfg', SAM2_MODEL_CFG] if SAM2_MODEL_CFG else []),
    '--target-columns', 'tumor',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', '1',
    '--num-workers', str(NUM_WORKERS),
    '--confidence-threshold', '0.5',
    '--cam-percentile', '85.0',
    '--max-points', '5',
    '--min-component-area', '40',
    '--mask-score-threshold', '0.4',
    '--selection-method', 'bone_hybrid',
    '--fusion-topk', str(FUSION_TOPK),
    *(['--disable-best-per-component'] if DISABLE_BEST_PER_COMPONENT else []),
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--max-bone-components', '6',
    '--points-per-component', '3',
    '--bbox-padding-ratio', '0.05',
    '--negative-points-per-component', '4',
    '--bone-seed-percentile', '82',
    '--bone-support-percentile', '78',
    '--closing-kernel', '5',
    '--opening-kernel', '0',
    '--max-hole-area', '200',
    '--min-size', '20',
    '--save-visuals-limit', '10',
    '--evaluate-prompt-quality',
    *process_args,
    '--output-dir', str(PSEUDO_OUTPUT),
]
print(' '.join(pseudo_cmd))
run_streaming(pseudo_cmd)

### 7.3 Prompt Quality: Foreground Localization and Point-Prompt Hit Rate

**Purpose:** `evaluate_ramh1200_masks.py` only measures the *final* pseudo mask, after
morphology + SAM + mask selection have all run — a low Dice there does not say whether
the failure came from a bad CAM, a badly placed prompt point, or a bad SAM/mask-selection
choice. This cell isolates the CAM/prompt stage using `prompt_quality.csv`
(written by `--evaluate-prompt-quality` above) against ground-truth tumor masks:

- **foreground_iou / foreground_recall / foreground_precision**: how well the actual
  region SAM is prompted from overlaps the true lesion. With the default
  `morphology-fusion-mode=components`, this is `bone_support`/`tumor_support` (the
  reconstructed support mask from seed+support thresholds), not a naive CAM percentile
  cut — it reflects what SAM really receives. Low recall means the foreground is missing
  part of the lesion; low precision means it extends into unrelated tissue.
- **point_hit_rate**: fraction of SAM prompt points that land inside the true lesion. Low
  hit rate means SAM is being pointed at the wrong place regardless of how good the
  foreground region looks overall.

Only images with a real GT mask (tumor images in this split) are included — normal images
have nothing to localize against.

In [ ]:
quality_csv = PSEUDO_OUTPUT / 'prompt_quality.csv'
if quality_csv.exists():
    quality_df = pd.read_csv(quality_csv)
    display(quality_df.describe())
    print(f"Mean foreground_iou: {quality_df['foreground_iou'].mean():.4f}")
    print(f"Mean foreground_recall: {quality_df['foreground_recall'].mean():.4f}")
    print(f"Mean foreground_precision: {quality_df['foreground_precision'].mean():.4f}")
    print(f"Mean point_hit_rate: {quality_df['point_hit_rate'].mean():.4f}")

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    quality_df['foreground_recall'].hist(ax=axes[0], bins=20)
    axes[0].set_title('Foreground recall (lesion coverage)')
    quality_df['foreground_precision'].hist(ax=axes[1], bins=20)
    axes[1].set_title('Foreground precision (localization accuracy)')
    quality_df['point_hit_rate'].hist(ax=axes[2], bins=20)
    axes[2].set_title('Point-prompt hit rate')
    plt.tight_layout()
    plt.show()

    # Worst-localized images are the most useful ones to inspect visually.
    worst = quality_df.sort_values('foreground_recall').head(5)
    print('Worst foreground_recall images (CAM/support missing most of the lesion):')
    display(worst)
else:
    print('prompt_quality.csv not found. Re-run Stage 2 with --evaluate-prompt-quality set (already default above).')

### 7.4 SAM-vs-Mask-Selection Oracle Diagnostic

**Purpose:** a low final Dice doesn't say *why* on its own — it could mean SAM never
proposed a good candidate mask for a lesion, or a good candidate existed but
`select_and_fuse_masks` (mask_selection.py) picked a worse one instead. This cell
answers that using `oracle_best_single_dice` / `selected_dice` / `oracle_gap_dice` from
`prompt_quality.csv` (written by `--evaluate-prompt-quality` above):

- **oracle_best_single_dice**: best Dice among all raw SAM candidates for that image —
  the ceiling on what mask selection could have produced from this exact candidate set.
- **selected_dice**: Dice of the mask `select_and_fuse_masks` actually chose (before
  Stage 6 morphological refinement).
- **oracle_gap_dice** = oracle − selected. A **large gap** means mask selection is
  discarding a good candidate that was already there — fix scoring/thresholds in
  `mask_selection.py`. A **small gap with a low oracle score** means SAM itself never
  produced a usable candidate for that lesion — fix prompts/support instead, not
  selection.

In [ ]:
if quality_csv.exists():
    oracle_df = pd.read_csv(quality_csv)
    has_oracle = oracle_df['oracle_best_single_dice'].notna()
    oracle_valid = oracle_df[has_oracle]

    if not oracle_valid.empty:
        mean_oracle = oracle_valid['oracle_best_single_dice'].mean()
        mean_selected = oracle_valid['selected_dice'].mean()
        mean_gap = oracle_valid['oracle_gap_dice'].mean()
        print(f"Images with oracle diagnostic: {len(oracle_valid)}")
        print(f"Mean oracle_best_single_dice: {mean_oracle:.4f}")
        print(f"Mean selected_dice: {mean_selected:.4f}")
        print(f"Mean gap (oracle - selected): {mean_gap:.4f}")
        if mean_gap > 0.1:
            print("=> Large gap: mask_selection.py is discarding good candidates on average.")
        elif mean_oracle < 0.3:
            print("=> Small gap but low oracle score: SAM/prompts rarely produce a good candidate.")
        else:
            print("=> Small gap and reasonable oracle score: selection is close to optimal already.")

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].scatter(oracle_valid['oracle_best_single_dice'], oracle_valid['selected_dice'], alpha=0.4, s=15)
        axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1)
        axes[0].set_xlabel('oracle_best_single_dice')
        axes[0].set_ylabel('selected_dice')
        axes[0].set_title('Selected vs. oracle-best candidate Dice')
        oracle_valid['oracle_gap_dice'].hist(ax=axes[1], bins=20)
        axes[1].set_title('Oracle gap distribution (oracle - selected)')
        plt.tight_layout()
        plt.show()

        # Images where selection discarded a much better candidate — worth inspecting.
        worst_gap = oracle_valid.sort_values('oracle_gap_dice', ascending=False).head(5)
        print('Worst selection gaps (a much better candidate existed but was not chosen):')
        display(worst_gap[['image_name', 'oracle_best_single_dice', 'selected_dice', 'oracle_gap_dice']])
    else:
        print('No oracle diagnostic values found (all NaN) — check --evaluate-prompt-quality is set.')
else:
    print('prompt_quality.csv not found. Re-run Stage 2 with --evaluate-prompt-quality set (already default above).')

### 7.5 Support-Loss vs Selection-Loss Decomposition

**Purpose:** section 7.4's `oracle_gap_dice` conflates two unrelated failure modes into
one number. `select_and_fuse_masks`'s `bone_hybrid` path always intersects its final
mask with `bone_support` (`constrain_to_bone_support`) before returning — `selected_dice`
already went through that clip, but the old `oracle_best_single_dice` never did, so a
large gap could mean either:

- **support loss**: `bone_support` (from the pre-SAM morphology stage) under-covers the
  true lesion, so even the *best* SAM candidate gets clipped down to near-nothing —
  independent of which candidate `mask_selection.py` picked. Fix the morphology stage
  (`--bone-seed-percentile` / `--bone-support-percentile`), not scoring.
- **selection loss**: `bone_hybrid` scoring picked a worse candidate than the best one
  available *after* the support clip. Fix `mask_selection.py`'s scoring/thresholds.

`oracle_best_single_dice_clipped` (added alongside the original oracle) applies the same
clip to every raw candidate individually and reports the best result — this is directly
comparable to `selected_dice`, so `oracle_best_single_dice_clipped - selected_dice` isolates
selection loss cleanly:

```text
oracle_best_single_dice              (raw candidate, no clip)
        │
        │  support_loss_dice = raw - clipped
        ▼
oracle_best_single_dice_clipped      (best candidate, clipped to bone_support)
        │
        │  selection_loss_dice = clipped - selected
        ▼
selected_dice                        (what mask_selection.py actually picked, already clipped)
```

**Caveat:** with the default `best_per_component=True`, `selected_dice` can come from a
*union* of several components' best candidates, which can occasionally score *higher*
than any single clipped candidate — `selection_loss_dice` can then be slightly negative.
That is not a bug; it means the union helped for that image. Large **positive**
`selection_loss_dice` is the real signal to act on.

In [ ]:
if quality_csv.exists():
    decomp_df = pd.read_csv(quality_csv)
    has_decomp = decomp_df['oracle_best_single_dice_clipped'].notna()
    decomp_valid = decomp_df[has_decomp]

    if not decomp_valid.empty:
        mean_raw = decomp_valid['oracle_best_single_dice'].mean()
        mean_clipped = decomp_valid['oracle_best_single_dice_clipped'].mean()
        mean_selected = decomp_valid['selected_dice'].mean()
        mean_support_loss = decomp_valid['support_loss_dice'].mean()
        mean_selection_loss = decomp_valid['selection_loss_dice'].mean()
        print(f"Images with decomposition: {len(decomp_valid)}")
        print(f"Mean oracle_best_single_dice (raw):     {mean_raw:.4f}")
        print(f"Mean oracle_best_single_dice_clipped:   {mean_clipped:.4f}")
        print(f"Mean selected_dice:                     {mean_selected:.4f}")
        print(f"Mean support_loss_dice (raw - clipped):     {mean_support_loss:.4f}")
        print(f"Mean selection_loss_dice (clipped - selected): {mean_selection_loss:.4f}")
        if mean_support_loss > mean_selection_loss and mean_support_loss > 0.05:
            print("=> Support loss dominates: bone_support under-covers lesions on average. "
                  "Fix --bone-seed-percentile/--bone-support-percentile, not mask_selection.py.")
        elif mean_selection_loss > 0.05:
            print("=> Selection loss dominates: bone_hybrid is discarding good clipped "
                  "candidates on average. Fix mask_selection.py scoring/thresholds.")
        else:
            print("=> Both losses are small: selection is close to optimal given the support it receives.")

        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].scatter(decomp_valid['oracle_best_single_dice'], decomp_valid['oracle_best_single_dice_clipped'], alpha=0.4, s=15)
        axes[0].plot([0, 1], [0, 1], 'r--', linewidth=1)
        axes[0].set_xlabel('oracle_best_single_dice (raw)')
        axes[0].set_ylabel('oracle_best_single_dice_clipped')
        axes[0].set_title('Support-clip effect on the best candidate')

        decomp_valid['support_loss_dice'].hist(ax=axes[1], bins=20)
        axes[1].set_title('Support loss distribution\n(raw - clipped)')

        decomp_valid['selection_loss_dice'].hist(ax=axes[2], bins=20)
        axes[2].set_title('Selection loss distribution\n(clipped - selected)')
        plt.tight_layout()
        plt.show()

        # Per-image breakdown, sorted by whichever loss is larger — the most useful
        # images to inspect visually for each failure mode.
        worst_support = decomp_valid.sort_values('support_loss_dice', ascending=False).head(5)
        print('Worst support loss (bone_support cut off most of a good candidate):')
        display(worst_support[[
            'image_name', 'oracle_best_single_dice', 'oracle_best_single_dice_clipped',
            'selected_dice', 'support_loss_dice', 'selection_loss_dice',
        ]])

        worst_selection = decomp_valid.sort_values('selection_loss_dice', ascending=False).head(5)
        print('Worst selection loss (a good clipped candidate existed but was not chosen):')
        display(worst_selection[[
            'image_name', 'oracle_best_single_dice', 'oracle_best_single_dice_clipped',
            'selected_dice', 'support_loss_dice', 'selection_loss_dice',
        ]])
    else:
        print('No support/selection decomposition values found (all NaN) — '
              're-run Stage 2 with the updated generate_pseudo_masks.py that passes '
              'bone_support into oracle_vs_selected_metrics.')
else:
    print('prompt_quality.csv not found. Re-run Stage 2 with --evaluate-prompt-quality set (already default above).')

### 7.1 Visual Check: Original Image, LayerCAM Overlay, and Pseudo Mask

Inspect this before running a full split. The CAM should activate around the tumor
region, and the pseudo mask should stay localized to the lesion rather than spreading
across the whole bone/limb.

In [ ]:
mask_dir = PSEUDO_OUTPUT / 'masks'
overlay_dir = PSEUDO_OUTPUT / 'overlays'
mask_files = sorted(mask_dir.glob('*.png'))
print('Number of pseudo masks:', len(mask_files), mask_dir)

n = min(5, len(mask_files))
if n:
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1:
        axes = np.array([axes])
    val_dir = BTXRD_ROOT / 'images'
    for row, mask_path in zip(axes, mask_files[:n]):
        stem = mask_path.stem
        image_match = next(iter(sorted(val_dir.glob(f'{stem}.*'))), None)
        overlay_match = next(iter(sorted(overlay_dir.glob(f'{stem}*fused_layercam.png'))), None)
        if image_match:
            row[0].imshow(Image.open(image_match).convert('RGB'))
            row[0].set_title(f'Original\n{stem}')
        if overlay_match:
            row[1].imshow(Image.open(overlay_match).convert('RGB'))
            row[1].set_title('LayerCAM overlay')
        row[2].imshow(Image.open(mask_path), cmap='gray')
        row[2].set_title('Pseudo mask')
        for ax in row:
            ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No pseudo masks were found.')

### 7.2 Image-Processing Explainer Before SAM

This cell visualizes the image-processing path before SAM, using `tumor_morphology.py`:

- original X-ray;
- LayerCAM;
- CAM + tumor guidance prompt map;
- enhanced grayscale;
- edge response;
- local intensity anomaly (catches both lytic/dark and sclerotic/bright lesions);
- tumor likelihood;
- seed pixels;
- morphology support;
- selected prompt points.

In [ ]:
if RUN_MORPHOLOGY_EXPLAINER and CLASSIFIER_CHECKPOINT.exists():
    import torch
    from datasets.common import make_classification_transform
    from models.classifier import DenseNet121AnatomyClassifier
    from models.layercam import LayerCAM
    from pseudo.generate_layercam import generate_fused_cam
    from pseudo.tumor_morphology import build_tumor_guidance, fuse_cam_with_bone_guidance
    from pseudo.extract_prompts import extract_point_prompts
    from pseudo.visualization import tensor_to_pil

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(CLASSIFIER_CHECKPOINT, map_location='cpu')
    target_columns = checkpoint.get('target_columns', ['tumor'])
    model = DenseNet121AnatomyClassifier(num_classes=len(target_columns), pretrained=False)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    model.to(device).eval()

    # Prefer a tumor-positive sample so the explainer shows a real lesion.
    explainer_index = next(i for i, s in enumerate(preview_ds.samples) if s['tumor'])
    explainer_name = preview_ds.samples[explainer_index]['image_id']
    explainer_image = preview_ds.images_dir / explainer_name
    image_pil = Image.open(explainer_image).convert('RGB')
    transform = make_classification_transform(IMAGE_SIZE, augment=False)
    image_tensor = transform(image_pil).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image_tensor)
        class_weights = torch.sigmoid(logits)[0].detach().cpu().numpy()
    print('Classifier tumor score:', float(class_weights[0]))

    layercam = LayerCAM(model, device=device)
    try:
        fused_cam, per_class_cams, active_indices = generate_fused_cam(
            layercam, image_tensor, class_weights=class_weights, confidence_threshold=0.3
        )
    finally:
        layercam.close()

    image_denorm = tensor_to_pil(image_tensor[0].detach().cpu())
    image_rgb = np.array(image_denorm, dtype=np.uint8)
    debug_dir = MORPH_DEBUG_OUTPUT / explainer_image.stem
    tumor_likelihood, tumor_support = build_tumor_guidance(
        image_rgb, fused_cam, seed_percentile=82, support_percentile=78,
        min_component_area=20, debug_dir=debug_dir,
    )
    prompt_map = fuse_cam_with_bone_guidance(fused_cam, tumor_likelihood, tumor_support)
    point_prompts = extract_point_prompts(prompt_map, cam_percentile=85, max_points=5, min_component_area=40, support_mask=tumor_support)

    panels = [
        ('Original', image_rgb, None),
        ('LayerCAM', fused_cam, 'jet'),
        ('CAM + tumor guidance', prompt_map, 'jet'),
        ('Enhanced grayscale', np.array(Image.open(debug_dir / 'tumor_gray_enhanced.png')), 'gray'),
        ('Edge response', np.array(Image.open(debug_dir / 'tumor_edge_response.png')), 'gray'),
        ('Local intensity anomaly', np.array(Image.open(debug_dir / 'tumor_anomaly_response.png')), 'gray'),
        ('Tumor likelihood', np.array(Image.open(debug_dir / 'tumor_likelihood.png')), 'gray'),
        ('Morphology support', np.array(Image.open(debug_dir / 'tumor_support.png')), 'gray'),
    ]
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.ravel()
    for ax, (title, arr, cmap) in zip(axes, panels):
        ax.imshow(arr, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    for r, c in point_prompts:
        axes[2].scatter([c], [r], c='red', s=70, edgecolors='white')
    plt.tight_layout()
    plt.show()
else:
    print('RUN_MORPHOLOGY_EXPLAINER=False or classifier checkpoint is missing.')

## 8. Quantitative Evaluation: Pseudo Mask vs BTXRD Ground Truth

**Purpose:** measure pseudo-mask quality against the LabelMe polygon-derived tumor
masks, reported as **two separate groups** rather than one pooled mean:

- **Tumor images — Dice/IoU**: this is the actual segmentation quality metric. A
  pooled mean over tumor *and* normal images is misleading here, because a normal
  image with an empty GT mask trivially scores Dice=1 when the pipeline also predicts
  an empty mask — that's a correct *detection* ("no lesion here"), not evidence of
  good *segmentation*. Mixing the two into one number inflates the headline metric
  without saying anything about how well real lesions are segmented.
- **Normal images — specificity / false positive rate**: specificity is the fraction
  of normal images where the pipeline correctly predicted nothing; false positive
  rate is `1 - specificity`. This is the correct way to report performance on images
  with no lesion.

With the default `RUN_FULL_PSEUDO_MASKS=True`, this reports metrics on the full
validation split. Set it to `False` while tuning parameters — it reports metrics on
just the 15-image sample from Stage 2 instead, which is enough to catch obviously bad
settings before spending time on the full split.

In [ ]:
eval_cmd = [
    sys.executable, 'evaluate_ramh1200_masks.py',
    '--dataset', DATASET_NAME,
    '--ram-root', str(BTXRD_ROOT),
    '--split', 'val',
    '--pred-mask-root', str(PSEUDO_OUTPUT / 'masks'),
    '--image-size', str(IMAGE_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--output-csv', str(EVAL_CSV),
]
print(' '.join(eval_cmd))
if RUN_EVALUATE_PSEUDO:
    run_streaming(eval_cmd)
else:
    print('RUN_EVALUATE_PSEUDO=False; evaluation is skipped.')

if EVAL_CSV.exists():
    eval_df = pd.read_csv(EVAL_CSV)
    display(eval_df.tail(10))

    # A single Mean Dice pooled over tumor + normal images conflates three
    # different things:
    #  - normal images trivially score Dice=1 when both prediction and GT are
    #    empty, which is a correct *detection* ("no lesion here"), not
    #    evidence of good *segmentation*;
    #  - tumor images the classifier skipped for low confidence always get an
    #    all-zero mask before CAM/SAM ever runs, so their Dice≈0 blames the
    #    classifier threshold, not CAM/SAM/morphology quality;
    #  - only tumor images that actually ran the full pipeline reflect
    #    CAM/SAM segmentation quality.
    # See evaluate_ramh1200_masks.py's group / skipped_low_confidence columns.
    ok_rows = eval_df[eval_df['status'] == 'ok'].copy()
    tumor_rows_all = ok_rows[ok_rows['group'] == 'tumor']
    tumor_skipped = tumor_rows_all[tumor_rows_all['skipped_low_confidence'] == True]
    tumor_rows = tumor_rows_all[tumor_rows_all['skipped_low_confidence'] == False]
    normal_rows = ok_rows[ok_rows['group'] == 'normal']

    if not tumor_rows_all.empty:
        print(f"Tumor images: {len(tumor_rows_all)} total, {len(tumor_skipped)} skipped for low classifier confidence")
    if not tumor_rows.empty:
        print(f"Tumor images evaluated (ran full pipeline): {len(tumor_rows)}")
        print(f"Mean Dice (tumor only): {tumor_rows['dice'].mean():.4f}")
        print(f"Mean IoU  (tumor only): {tumor_rows['iou'].mean():.4f}")
    if not normal_rows.empty:
        empty_pred_rate = (normal_rows['dice'] >= 0.999).mean()
        print(f"Normal images evaluated: {len(normal_rows)}")
        print(f"Specificity (correctly empty pseudo-mask rate): {empty_pred_rate:.4f}")
        print(f"False positive rate: {1 - empty_pred_rate:.4f}")

    if not tumor_rows.empty:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        tumor_rows['dice'].hist(ax=axes[0], bins=20)
        axes[0].set_title('Pseudo-mask Dice distribution (tumor images only)')
        tumor_rows['iou'].hist(ax=axes[1], bins=20)
        axes[1].set_title('Pseudo-mask IoU distribution (tumor images only)')
        plt.tight_layout()
        plt.show()

## 9. Stage 3: Supervised U-Net Baseline

**Pipeline role:** train a fully supervised U-Net using BTXRD ground-truth tumor masks.

This gives a supervised reference point for the pseudo-mask branch.

In [ ]:
# Stop early if val_dice does not improve for this many consecutive epochs (0 disables).
UNET_EARLY_STOP_PATIENCE = 7

unet_cmd = [
    sys.executable, 'train_segmentation.py',
    '--dataset', DATASET_NAME,
    '--ram-root', str(BTXRD_ROOT),
    '--train-split', 'train',
    '--val-split', 'val',
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE_SEGMENTATION),
    '--num-workers', str(NUM_WORKERS),
    '--epochs', str(EPOCHS_SEGMENTATION),
    '--output-dir', str(SEG_OUTPUT),
    '--early-stop-patience', str(UNET_EARLY_STOP_PATIENCE),
]
print(' '.join(unet_cmd))
if RUN_TRAIN_UNET:
    run_streaming(unet_cmd)
else:
    print('RUN_TRAIN_UNET=False; U-Net training is skipped.')

seg_log = SEG_OUTPUT / 'training_log.csv'
seg_ckpt = SEG_OUTPUT / 'best_unet.pt'
print('U-Net checkpoint:', seg_ckpt, 'exists=', seg_ckpt.exists())
if seg_log.exists():
    seg_df = pd.read_csv(seg_log)
    display(seg_df.tail())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    seg_df[['train_loss', 'val_loss']].plot(ax=axes[0], title='U-Net loss')
    seg_df[['train_dice', 'val_dice']].plot(ax=axes[1], title='U-Net Dice')
    seg_df[['train_iou', 'val_iou']].plot(ax=axes[2], title='U-Net IoU')
    for ax in axes:
        ax.set_xlabel('epoch')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 10. Full Pipeline Visualization on One Image

This stage creates a pipeline strip with `visualize_pipeline.py`: original image,
LayerCAM, tumor guidance, prompts, SAM candidates, pseudo mask, and optionally U-Net
output.

In [ ]:
sample_index = next(i for i, s in enumerate(preview_ds.samples) if s['tumor'])
sample_image = preview_ds.images_dir / preview_ds.samples[sample_index]['image_id']
print('Sample image:', sample_image)

viz_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--dataset', DATASET_NAME,
    '--image-path', str(sample_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--sam-version', SAM_VERSION,
    *(['--sam2-model-cfg', SAM2_MODEL_CFG] if SAM2_MODEL_CFG else []),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--output-path', str(VIZ_OUTPUT / f'{sample_image.stem}_pipeline.png'),
]
if (SEG_OUTPUT / 'best_unet.pt').exists():
    viz_cmd.extend(['--segmentation-checkpoint', str(SEG_OUTPUT / 'best_unet.pt')])
print(' '.join(viz_cmd))
if RUN_VISUALIZE_SAMPLE and sample_image is not None:
    run_streaming(viz_cmd)

viz_files = sorted(VIZ_OUTPUT.glob('*_pipeline.png'))
if viz_files:
    img = Image.open(viz_files[-1])
    plt.figure(figsize=(22, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(viz_files[-1].name)
    plt.show()

## 11. SAM Candidate Debugging

This is the failure-analysis stage. It saves and displays every SAM candidate mask,
each overlay, and `scores.json`.

In [ ]:
debug_image = sample_image
DEBUG_OUTPUT.mkdir(parents=True, exist_ok=True)
debug_strip = DEBUG_OUTPUT / f'{debug_image.stem}_debug_pipeline.png'

debug_cmd = [
    sys.executable, 'visualize_pipeline.py',
    '--dataset', DATASET_NAME,
    '--image-path', str(debug_image),
    '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
    '--sam-checkpoint', str(SAM_CHECKPOINT),
    '--sam-version', SAM_VERSION,
    *(['--sam2-model-cfg', SAM2_MODEL_CFG] if SAM2_MODEL_CFG else []),
    '--image-size', str(IMAGE_SIZE),
    '--selection-method', 'bone_hybrid',
    '--morphology-fusion-mode', 'components',
    '--sam-prompt-mode', 'box_point',
    '--debug',
    '--output-path', str(debug_strip),
]
print(' '.join(debug_cmd))
if RUN_DEBUG_SAM:
    run_streaming(debug_cmd)

if debug_strip.exists():
    plt.figure(figsize=(22, 4))
    plt.imshow(Image.open(debug_strip))
    plt.axis('off')
    plt.title('Debug pipeline strip')
    plt.show()

import json
debug_dir = DEBUG_OUTPUT / 'debug' / debug_image.stem
mask_files = sorted(debug_dir.glob('mask_*.png'))
overlay_files = sorted(debug_dir.glob('overlay_mask_*.png'))
scores_path = debug_dir / 'scores.json'
print('debug_dir:', debug_dir)
print('SAM candidate masks:', len(mask_files))

if mask_files:
    n = len(mask_files)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
    if n == 1:
        axes = np.array([[axes[0]], [axes[1]]])
    for i, mf in enumerate(mask_files):
        axes[0][i].imshow(Image.open(mf), cmap='gray')
        axes[0][i].set_title(mf.stem)
        axes[0][i].axis('off')
        if i < len(overlay_files):
            axes[1][i].imshow(Image.open(overlay_files[i]).convert('RGB'))
        axes[1][i].set_title('overlay')
        axes[1][i].axis('off')
    plt.tight_layout()
    plt.show()

if scores_path.exists():
    scores = json.loads(scores_path.read_text(encoding='utf-8'))
    print('scores.json')
    for key, value in scores.items():
        score = value.get('score', 0)
        area = value.get('area', 0)
        bar = '#' * int(score * 30)
        print(f'{key:<8} score={score:.3f} area={area:>8} {bar}')

## 12. Small Ablation Preview

Optional. Enable `RUN_ABLATION_PREVIEW=True` to compare a few prompt/scoring
strategies on the same image.

In [ ]:
ABLATION_OUTPUT.mkdir(parents=True, exist_ok=True)
ABLATION_STRATEGIES = [
    ('box_point_bone_hybrid', ['--sam-prompt-mode', 'box_point', '--selection-method', 'bone_hybrid']),
    ('point_bone_hybrid', ['--sam-prompt-mode', 'point', '--selection-method', 'bone_hybrid']),
    ('box_point_coverage', ['--sam-prompt-mode', 'box_point', '--selection-method', 'coverage']),
]

if RUN_ABLATION_PREVIEW:
    for label, extra in ABLATION_STRATEGIES:
        out_path = ABLATION_OUTPUT / f'{label}_pipeline.png'
        cmd = [
            sys.executable, 'visualize_pipeline.py',
            '--dataset', DATASET_NAME,
            '--image-path', str(debug_image),
            '--classifier-checkpoint', str(CLASSIFIER_CHECKPOINT),
            '--sam-checkpoint', str(SAM_CHECKPOINT),
            '--sam-version', SAM_VERSION,
            *(['--sam2-model-cfg', SAM2_MODEL_CFG] if SAM2_MODEL_CFG else []),
            '--image-size', str(IMAGE_SIZE),
            '--morphology-fusion-mode', 'components',
            '--output-path', str(out_path),
            *extra,
        ]
        print(' '.join(cmd))
        run_streaming(cmd)

ablation_files = sorted(ABLATION_OUTPUT.glob('*_pipeline.png'))
if ablation_files:
    fig, axes = plt.subplots(len(ablation_files), 1, figsize=(22, 4*len(ablation_files)))
    if len(ablation_files) == 1:
        axes = [axes]
    for ax, path in zip(axes, ablation_files):
        ax.imshow(Image.open(path))
        ax.set_title(path.stem)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 13. Reporting Checklist

Save these artifacts for the thesis/report:

1. Dataset sample: original X-ray + GT tumor mask + overlay, for both a tumor and a
   normal case.
2. Split table: image/tumor/malignant counts per train/val/test.
3. Stage 2 sample: original + LayerCAM overlay + pseudo mask.
4. Morphology explainer: grayscale, edge, local anomaly, likelihood, seeds, support,
   prompt points.
5. Pipeline strip from `visualize_pipeline.py`.
6. Dice/IoU table from `evaluate_ramh1200_masks.py --dataset btxrd`.
7. Supervised U-Net training curves and best validation Dice/IoU.
8. Optional ablation figures for prompt mode or scoring method.
9. Optional: compare BTXRD Dice/IoU against the RAM-H1200 run to discuss how the
   tumor-vs-bone target and morphology prior affect pseudo-mask quality.